# Imports

In [1]:
# !python3 -m venv venv
!source venv/bin/activate

In [2]:
# !pip install matplotlib

# !pip install scvi-tools
# !pip install scanpy
# !pip install torch torchvision torchaudio
# # if only CPU available
# # !pipx install jax
# # if Nvidia GPU available
# !pip install "jax[cuda12]"

# !pip install numpy==1.26.4

In [6]:
import os

import scanpy as sc
import anndata as ad
import scvi

# Subset GongSharma data

In [4]:
folder_path = os.path.join(os.getcwd(), "GongSharma_data")
files_in_folder = os.listdir(folder_path)
files_in_folder.remove(".ipynb_checkpoints")
files_in_folder

['Sound_Life_YoungAdult_Female_CMVneg.h5ad',
 'Sound_Life_OlderAdult_Female_CMVneg.h5ad',
 'Sound_Life_OlderAdult_Female_CMVpos.h5ad',
 'Sound_Life_YoungAdult_Male_CMVneg.h5ad',
 'Sound_Life_OlderAdult_Male_CMVpos.h5ad',
 'Sound_Life_YoungAdult_Male_CMVpos.h5ad',
 'Sound_Life_YoungAdult_Female_CMVpos.h5ad',
 'Sound_Life_OlderAdult_Male_CMVneg.h5ad']

In [5]:
female_files = [s for s in files_in_folder if "Female" in s]
female_files

['Sound_Life_YoungAdult_Female_CMVneg.h5ad',
 'Sound_Life_OlderAdult_Female_CMVneg.h5ad',
 'Sound_Life_OlderAdult_Female_CMVpos.h5ad',
 'Sound_Life_YoungAdult_Female_CMVpos.h5ad']

In [7]:
adata_path = os.path.join(folder_path, 'Sound_Life_YoungAdult_Female_CMVneg.h5ad')
adata = sc.read_h5ad(adata_path, backed='r+')

In [8]:
adata

AnnData object with n_obs × n_vars = 2616824 × 33538 backed at '/home/bronzen/analysis_ZhangY_2022_BRCA/python/GongSharma_data/Sound_Life_YoungAdult_Female_CMVneg.h5ad'
    obs: 'barcodes', 'original_barcodes', 'cell_name', 'batch_id', 'pool_id', 'chip_id', 'well_id', 'n_genes', 'n_reads', 'n_umis', 'total_counts_mito', 'pct_counts_mito', 'doublet_score', 'predicted_AIFI_L1', 'AIFI_L1_score', 'AIFI_L1', 'predicted_AIFI_L2', 'AIFI_L2_score', 'AIFI_L2', 'predicted_AIFI_L3', 'AIFI_L3_score', 'AIFI_L3', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.cmv', 'subject.bmi', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'subject.ageAtFirstDraw', 'sample.visitName', 'sample.drawDate', 'sample.subjectAgeAtDraw', 'specimen.specimenGuid', 'pipeline.fileGuid', 'subject.ageGroup'

In [6]:
# Subset samples for comparison CMV pos vs neg (~30 samples, so 8 from 4 total groups)
num_samples = 8

output_file_name = os.path.join(os.getcwd(), "data", "Gongsharma_cmv_females")

if not os.path.exists(output_file_name):
    for h5ad_file in female_files:
        adata_path = os.path.join(folder_path, h5ad_file)
        adata = sc.read_h5ad(adata_path, backed='r+')
    
        # Group by individual and select the first unique sample name
        first_sample_per_individual = adata.obs.groupby('subject.subjectGuid')['specimen.specimenGuid'].first()
    
        # Subset
        mask = adata.obs["specimen.specimenGuid"].isin(first_sample_per_individual[:num_samples])
    
        # Append bdata to combined_adata
        if h5ad_file == files_in_folder[0]:
            combined_adata = adata[mask]
        else:
            combined_adata = ad.concat([combined_adata, adata[mask]])

    combined_adata.obs['Sample'] = combined_adata.obs['specimen.specimenGuid']
    
    combined_adata.write_h5ad(output_file_name + '.h5ad')
    combined_adata.obs.to_feather(output_file_name + '.feather')

/tmp/ipykernel_2830/3482237472.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  first_sample_per_individual = adata.obs.groupby('subject.subjectGuid')['specimen.specimenGuid'].first()
/tmp/ipykernel_2830/3482237472.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  first_sample_per_individual = adata.obs.groupby('subject.subjectGuid')['specimen.specimenGuid'].first()
/tmp/ipykernel_2830/3482237472.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future 

In [7]:
female_cmvneg_files = [s for s in files_in_folder if "Female_CMVneg" in s]
female_cmvneg_files

['Sound_Life_YoungAdult_Female_CMVneg.h5ad',
 'Sound_Life_OlderAdult_Female_CMVneg.h5ad']

In [8]:
# Subset samples for comparison CMV pos vs neg (~30 samples, so 16 from 2 total groups)
num_samples = 16

output_file_name = os.path.join(os.getcwd(), "data", "Gongsharma_age_females_cmvneg")

if not os.path.exists(output_file_name):
    for h5ad_file in female_cmvneg_files:
        adata_path = os.path.join(folder_path, h5ad_file)
        adata = sc.read_h5ad(adata_path, backed='r+')
    
        # Group by individual and select the first unique sample name
        first_sample_per_individual = adata.obs.groupby('subject.subjectGuid')['specimen.specimenGuid'].first()
    
        # Subset
        mask = adata.obs["specimen.specimenGuid"].isin(first_sample_per_individual[:num_samples])
    
        # Append bdata to combined_adata
        if h5ad_file == files_in_folder[0]:
            combined_adata = adata[mask]
        else:
            combined_adata = ad.concat([combined_adata, adata[mask]])

    combined_adata.obs['Sample'] = combined_adata.obs['specimen.specimenGuid']
    
    combined_adata.write_h5ad(output_file_name + '.h5ad')
    combined_adata.obs.to_feather(output_file_name + '.feather')

/tmp/ipykernel_2830/1660476103.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  first_sample_per_individual = adata.obs.groupby('subject.subjectGuid')['specimen.specimenGuid'].first()
/tmp/ipykernel_2830/1660476103.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  first_sample_per_individual = adata.obs.groupby('subject.subjectGuid')['specimen.specimenGuid'].first()
